# Notebook 03 — BGE-M3 Text Embeddings & ChromaDB Index

**Phase 2 · Task group 113.** Goal: take the Tier-1 `chunks.parquet` from Notebook 02, embed every chunk with **BAAI/bge-m3** (1024-d), persist to a tier-namespaced ChromaDB collection, and produce evidence that BGE-M3 gives us a cleaner semantic space than the 2022 DPR baseline.

### What this notebook ships
1. Cache-or-build embedder — never re-embeds on a warm kernel / re-run.
2. Persistent Chroma collection `sciret_<tier>_bge_m3_cs400_o50`.
3. UMAP visualisation of the embedding space, coloured by publication year bucket.
4. Head-to-head BGE-M3 vs DPR similarity test on a curated prompt bank.
5. Semantic-similarity sanity test: `related_sim > unrelated_sim` for 3 pairs.

> Runtime note: on CPU, encoding 1k chunks takes ~5–8 min; on Kaggle GPU for 50k chunks closer to 15–25 min. The *same code path* runs both; only the tier flips.


In [1]:
import os, sys, json, warnings, time
from pathlib import Path
from typing import List

NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent if (NB_DIR.parent / "2_src").exists() else NB_DIR
sys.path.insert(0, str(PROJECT_ROOT / "2_src"))

os.environ.setdefault("SCIRET_TIER", "tier1")
from config import (
    get_config, BGE_M3_MODEL, BGE_M3_DIM,
    DPR_QUESTION_MODEL, DPR_CONTEXT_MODEL, DPR_DIM, SEED,
)
CFG = get_config()
print(CFG.summary())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore")


[SciRet:tier1] size=1000 root=/Users/kaysarulanasapurba/Desktop/SciRet chunks=chunks.parquet chroma=chroma_db/sciret_tier1_bge_m3_cs400_o50


In [2]:
chunks = pd.read_parquet(CFG.chunks_path)
print(f"chunks: {len(chunks):,}")
chunks.head(2)


chunks: 1,042


,chunk_id,cord_uid,chunk_index,title,chunk_text,n_tokens
0,3fmbrp7f_c000,3fmbrp7f,0,Ocular-symptoms-related Google Search Trends d...,Ocular-symptoms-related Google Search Trends d...,240
1,pnr3q5nr_c000,pnr3q5nr,0,Strength of Religious Faith in Peruvian Adoles...,Strength of Religious Faith in Peruvian Adoles...,214


## 1. Load BGE-M3

We use `sentence-transformers` for a single clean encode API. The model is downloaded once to the HF cache — subsequent runs (including Kaggle) pull from cache.

In [3]:
import torch
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)

model = SentenceTransformer(BGE_M3_MODEL, device=DEVICE)
test_vec = model.encode(["sanity check"], normalize_embeddings=True)
assert test_vec.shape[1] == BGE_M3_DIM, f"expected dim {BGE_M3_DIM}, got {test_vec.shape[1]}"
print(f"BGE-M3 ok — dim={test_vec.shape[1]}")


device: mps
BGE-M3 ok — dim=1024


## 2. Cache-or-build embeddings

The cached parquet holds `chunk_id` + raw vector so we can rebuild the Chroma collection from scratch without re-encoding.

In [ ]:
CACHE_PATH = CFG.embeddings_dir / f"bge_m3_{CFG.tier}.parquet"
MANIFEST_PATH = CFG.embeddings_dir / f"bge_m3_{CFG.tier}_manifest.json"

# def encode_chunks(df, batch=64):
def encode_chunks(df, batch=16): # original: batch=64
    texts = df["chunk_text"].fillna("").tolist()
    vecs = []
    t0 = time.time()
    for i in range(0, len(texts), batch):
        vecs.append(model.encode(
            texts[i:i+batch],
            normalize_embeddings=True,
            show_progress_bar=False,
            convert_to_numpy=True,
        ))
        if (i // batch) % 10 == 0:
            done = min(i + batch, len(texts))
            print(f"  {done:,}/{len(texts):,}  ({(time.time()-t0):.1f}s)")
    return np.vstack(vecs).astype("float32")

if CACHE_PATH.exists():
    emb_df = pd.read_parquet(CACHE_PATH)
    print(f"cache HIT  ->  {CACHE_PATH}   ({len(emb_df):,} vectors)")
else:
    vecs = encode_chunks(chunks)
    emb_df = pd.DataFrame({"chunk_id": chunks["chunk_id"].tolist(), "vector": list(vecs)})
    emb_df.to_parquet(CACHE_PATH, index=False)
    MANIFEST_PATH.write_text(json.dumps({
        "model_name": BGE_M3_MODEL, "dim": BGE_M3_DIM,
        "count": int(len(emb_df)), "tier": CFG.tier,
    }, indent=2))
    print(f"cache MISS -> saved {CACHE_PATH}")

assert len(emb_df) == len(chunks), "embedding count != chunk count"
vecs = np.asarray(emb_df["vector"].tolist(), dtype="float32")
print("vectors:", vecs.shape)


  64/1,042  (612.3s)


RuntimeError: MPS backend out of memory (MPS allocated: 7.72 GB, other allocations: 704.00 KB, max allowed: 9.07 GB). Tried to allocate 2.60 GB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

## 3. Persist into a tier-namespaced ChromaDB collection

In [ ]:
import chromadb
from chromadb.config import Settings

client = chromadb.PersistentClient(path=str(CFG.chroma_dir))
col_name = CFG.text_collection
print("collection name:", col_name)

try:
    col = client.get_collection(col_name)
    if col.count() == len(chunks):
        print(f"Chroma collection already populated ({col.count():,}) — skipping add.")
    else:
        print(f"existing count={col.count()} != chunks={len(chunks)} -> rebuilding")
        client.delete_collection(col_name)
        col = client.create_collection(col_name, metadata={"hnsw:space": "cosine"})
except Exception:
    col = client.create_collection(col_name, metadata={"hnsw:space": "cosine"})

if col.count() == 0:
    BATCH = 512
    ids = chunks["chunk_id"].astype(str).tolist()
    metas = chunks[["cord_uid","title","chunk_index","n_tokens"]].astype({"chunk_index":"int","n_tokens":"int"}).to_dict("records")
    docs = chunks["chunk_text"].tolist()
    for i in range(0, len(ids), BATCH):
        col.add(
            ids=ids[i:i+BATCH],
            embeddings=vecs[i:i+BATCH].tolist(),
            metadatas=metas[i:i+BATCH],
            documents=docs[i:i+BATCH],
        )
    print(f"Chroma populated: {col.count():,}")


## 4. UMAP — does BGE-M3 recover semantic clusters?

In [ ]:
# UMAP is optional; we fall back to PCA if umap-learn isn't installed.
try:
    import umap
    reducer = umap.UMAP(n_components=2, random_state=SEED, metric="cosine", n_neighbors=30, min_dist=0.1)
    reduce_name = "UMAP"
except Exception:
    from sklearn.decomposition import PCA
    reducer = PCA(n_components=2, random_state=SEED)
    reduce_name = "PCA"

SUB = min(2000, len(vecs))
idx = np.random.RandomState(SEED).choice(len(vecs), SUB, replace=False)
coords = reducer.fit_transform(vecs[idx])

meta_sub = chunks.iloc[idx].merge(
    pd.read_parquet(CFG.papers_clean_path)[["cord_uid","publish_year"]],
    on="cord_uid", how="left"
)
buckets = pd.cut(meta_sub["publish_year"].fillna(0),
                 bins=[-1,2019,2020,2021,2022,2100],
                 labels=["pre-2020","2020","2021","2022","2023+"])

fig, ax = plt.subplots(figsize=(7,6))
for b, color in zip(buckets.cat.categories, sns.color_palette("viridis", len(buckets.cat.categories))):
    mask = (buckets == b).values
    ax.scatter(coords[mask,0], coords[mask,1], s=6, alpha=0.6, color=color, label=str(b))
ax.set_title(f"{reduce_name} of BGE-M3 chunk embeddings (tier={CFG.tier}, n={SUB:,})")
ax.legend(title="publish year", loc="best", fontsize=8)
ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()
fig.savefig(CFG.results_dir / "umap_embeddings.png", dpi=140)
plt.show()


## 5. BGE-M3 vs DPR head-to-head

We pull DPR once, encode the same 500 chunk sample with both, and for a prompt bank of COVID-relevant queries compute `cos(query, chunk)`. For each (query, chunk) pair we record which model ranks the chunk higher.

In [ ]:
try:
    from transformers import DPRQuestionEncoder, DPRContextEncoder, AutoTokenizer
    q_enc = DPRQuestionEncoder.from_pretrained(DPR_QUESTION_MODEL).to(DEVICE).eval()
    c_enc = DPRContextEncoder.from_pretrained(DPR_CONTEXT_MODEL).to(DEVICE).eval()
    q_tok = AutoTokenizer.from_pretrained(DPR_QUESTION_MODEL)
    c_tok = AutoTokenizer.from_pretrained(DPR_CONTEXT_MODEL)
    dpr_ok = True
except Exception as e:
    print("DPR unavailable — skipping A/B:", e)
    dpr_ok = False


In [ ]:
QUERIES = [
    "What imaging techniques were used to study COVID-19 lung damage?",
    "How effective are mRNA vaccines against the Delta variant?",
    "What are the cardiac complications of COVID-19?",
    "Which antiviral drugs showed efficacy against SARS-CoV-2 in vitro?",
    "How is long COVID defined and what are its symptoms?",
    "What role does the ACE2 receptor play in SARS-CoV-2 infection?",
    "Describe the effectiveness of masks in preventing viral transmission.",
    "How did lockdowns affect mental health outcomes?",
]

def dpr_encode(texts, enc, tok, max_len=256):
    import torch
    outs = []
    for i in range(0, len(texts), 16):
        batch = texts[i:i+16]
        toks = tok(batch, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            v = enc(**toks).pooler_output.cpu().numpy()
        # L2-normalise
        v = v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-9)
        outs.append(v)
    return np.vstack(outs).astype("float32")

if dpr_ok:
    sample = chunks.sample(500, random_state=SEED).reset_index(drop=True)
    # BGE side (reuse cached vectors to save time)
    bge_sample = np.asarray(
        emb_df.set_index("chunk_id").loc[sample["chunk_id"].tolist(), "vector"].tolist(),
        dtype="float32",
    )
    bge_q = model.encode(QUERIES, normalize_embeddings=True)
    # DPR side
    dpr_sample = dpr_encode(sample["chunk_text"].tolist(), c_enc, c_tok)
    dpr_q = dpr_encode(QUERIES, q_enc, q_tok)

    wins_bge, wins_dpr, ties = 0, 0, 0
    per_query = []
    for qi in range(len(QUERIES)):
        sb = bge_sample @ bge_q[qi]
        sd = dpr_sample @ dpr_q[qi]
        rb = np.argsort(-sb); rd = np.argsort(-sd)
        # top-10 overlap
        overlap = len(set(rb[:10]) & set(rd[:10]))
        # winner = model with higher top-1 score
        if sb.max() > sd.max(): wins_bge += 1
        elif sd.max() > sb.max(): wins_dpr += 1
        else: ties += 1
        per_query.append({"query": QUERIES[qi][:60]+"...", "bge_top1": float(sb.max()),
                          "dpr_top1": float(sd.max()), "top10_overlap": overlap})
    pq = pd.DataFrame(per_query)
    print(f"wins  BGE-M3={wins_bge}  DPR={wins_dpr}  ties={ties}")
    display(pq.round(3))


In [ ]:
if dpr_ok:
    fig, ax = plt.subplots(figsize=(7,3.5))
    x = np.arange(len(pq))
    ax.bar(x-0.2, pq["bge_top1"], 0.4, label="BGE-M3", color="#2563eb")
    ax.bar(x+0.2, pq["dpr_top1"], 0.4, label="DPR (2022)", color="#f97316")
    ax.set_xticks(x); ax.set_xticklabels([f"Q{i+1}" for i in x])
    ax.set_ylabel("cos(top-1 chunk)")
    ax.set_title("BGE-M3 vs DPR — top-1 similarity per query")
    ax.legend()
    fig.tight_layout()
    fig.savefig(CFG.results_dir / "bge_vs_dpr.png", dpi=140)
    pq.to_csv(CFG.results_dir / "bge_vs_dpr.csv", index=False)
    plt.show()


## 6. Semantic similarity sanity test

For each `(related, unrelated)` pair we expect `sim(q, related) > sim(q, unrelated)`. If that gap ever goes negative, the embedder has a problem.

In [ ]:
PAIRS = [
    ("CT scans for COVID-19 lung pneumonia",
     "radiological imaging of SARS-CoV-2 infection in the thorax",
     "machine learning survey on recommender systems"),
    ("ACE2 receptor binding of the spike protein",
     "spike glycoprotein engages ACE2 to enter host cells",
     "effects of exercise on cardiovascular endurance"),
    ("mRNA vaccine efficacy against variants",
     "BNT162b2 and mRNA-1273 neutralisation of Omicron",
     "climate change impact on agricultural yield"),
]

rows = []
for q, rel, un in PAIRS:
    vq, vr, vu = model.encode([q, rel, un], normalize_embeddings=True)
    s_rel = float(np.dot(vq, vr))
    s_un  = float(np.dot(vq, vu))
    rows.append({"query": q[:35]+"…", "related": round(s_rel,3), "unrelated": round(s_un,3),
                 "gap": round(s_rel - s_un, 3)})
sim_df = pd.DataFrame(rows)
sim_df


In [ ]:
assert (sim_df["gap"] > 0).all(), "Embedder failed semantic sanity — investigate."
print("sanity OK  — all related > unrelated")
sim_df.to_csv(CFG.results_dir / "bge_semantic_sanity.csv", index=False)


---
**Outputs**
* `1_data/embeddings/<tier>/bge_m3_<tier>.parquet` — cached vectors
* `1_data/embeddings/<tier>/chroma_db/sciret_<tier>_bge_m3_cs400_o50` — persistent Chroma collection
* `4_results/<tier>/umap_embeddings.png`
* `4_results/<tier>/bge_vs_dpr.{png,csv}`
* `4_results/<tier>/bge_semantic_sanity.csv`

**Next:** Notebook 04 — BM25 + dense + RRF hybrid retrieval.
